In [ ]:
from utils import globals

import pandas as pd
import os
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import seaborn as sns
import plotly.graph_objects as go
from plotly.offline import iplot
from scipy.fftpack import fft
from importlib import reload

In [ ]:
orig = globals.load_training()
train = orig.dropna().copy() #757 righe in meno

# Punti in cui avvengono le manutenzioni
wws = globals.get_shift('Cumulative_WWs', train)
hpc = globals.get_shift('Cumulative_HPC_SVs', train)
hpt = globals.get_shift('Cumulative_HPT_SVs', train)

In [ ]:

reload(globals)
ESN=101
SENSOR = globals.SENSORS.Sensed_Pamb.value
filtered = globals.filter(train, "ESN", ESN, [SENSOR, 'Snapshot'])
wsize = 8
rolling_mean = filtered[SENSOR].rolling(window=wsize).mean()

# Plotting
plt.figure(figsize=(10, 5))
plt.vlines(wws.loc[wws['ESN'] == ESN].index, ymin=filtered[SENSOR].min(), ymax=filtered[SENSOR].max(), colors='red', linestyles='dashed', label='WWs', alpha=0.7)
plt.plot(train.loc[train['ESN'] == int(ESN), [SENSOR]], label='Raw Data', color='blue', alpha=0.3)
plt.plot(rolling_mean, label='Rolling Mean', color='blue')
plt.title(f'{SENSOR} - Rolling Window (Size defined by Snapshots: {wsize})')
plt.xlabel('Cycles')
plt.ylabel('Sensor Value')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.savefig(f"{globals.PLOT_PATH}/ROLLING-{SENSOR}-{wsize}-ESN{ESN}")
plt.show()


In [ ]:
reload(globals)

ESN=104
points = hpt
plabel = "HPT"
method = "ROLLINGW"

path = f"{globals.PLOT_PATH}/{method}/{ESN}/"
os.makedirs(os.path.dirname(path), exist_ok=True)
for sensor in globals.SENSORS.iter():
    SENSOR = sensor
    filtered = globals.filter(train, "ESN", ESN, [SENSOR, 'Snapshot'])
    wsize = 8
    rolling_mean = filtered[SENSOR].rolling(window=wsize).mean()

# Plotting
    plt.figure(figsize=(10, 5))
    plt.vlines(points.loc[points['ESN'] == ESN].index, ymin=filtered[SENSOR].min(), ymax=filtered[SENSOR].max(), colors='red', linestyles='dashed', label=plabel, alpha=0.7)
    plt.plot(train.loc[train['ESN'] == int(ESN), [SENSOR]], label='Raw Data', color='blue', alpha=0.3)
    plt.plot(rolling_mean, label='Rolling Mean', color='blue')
    plt.title(f'{SENSOR} - Rolling Window (Size defined by Snapshots: {wsize})')
    plt.xlabel('Cycles')
    plt.ylabel('Sensor Value')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    
    plt.savefig(f"{path}/{SENSOR}-{plabel}-{wsize}")



In [ ]:
reload(globals)

ESN=101
method = "BY_SNAPSHOT"
phase = 1
points = hpt
plabel = "HPT"

path = f"{globals.PLOT_PATH}/{method}/{ESN}/"
os.makedirs(os.path.dirname(path), exist_ok=True)
for sensor in globals.SENSORS.iter():
    SENSOR = sensor
    filtered = globals.filter(train, "ESN", ESN, [SENSOR, 'Snapshot'])

    colors = cm.viridis(np.linspace(0, 1, 8))
    plt.figure(figsize=(50, 20))
    for i in range(1,9):
        df = globals.filter(filtered, "Snapshot", i, [SENSOR, 'Snapshot'])
        plt.plot(df, label=f'Valore nella fase {i} ',color=colors[i-1], alpha=0.4)

    plt.vlines(points.loc[points['ESN'] == ESN].index, ymin=filtered[SENSOR].min(), ymax=filtered[SENSOR].max(), colors='red', linestyles='dashed', label=plabel, alpha=0.7)
    plt.title(f'valore {SENSOR} per fase')
    plt.xlabel('Cycles')
    plt.ylabel('Sensor Value')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show() 
    plt.savefig(f"{path}/{SENSOR}-{plabel}")


In [ ]:
reload(globals)
import plotly.graph_objects as go
import os

ESN = 101
method = "BY_SNAPSHOT"
phase = 1
points = hpt
plabel = "HPT"

path = f"{globals.PLOT_PATH}/{method}/{ESN}/"
os.makedirs(path, exist_ok=True)

# Define a discrete colorscale (replaces cm.viridis)
colors = [
    '#440154', '#482878', '#3e4989', '#31688e', 
    '#26828e', '#1f9e89', '#6ece58', '#fde725'
]

for sensor in globals.SENSORS.iter():
    SENSOR = sensor
    filtered = globals.filter(train, "ESN", ESN, [SENSOR, 'Snapshot'])
    
    # Initialize the figure
    fig = go.Figure()

    # 1. Add the lines for each Snapshot
    for i in range(1, 9):
        df = globals.filter(filtered, "Snapshot", i, [SENSOR, 'Snapshot'])
        
        fig.add_trace(go.Scatter(
            x=df.index, 
            y=df[SENSOR],
            mode='lines',
            name=f'Fase {i}',
            line=dict(color=colors[i-1], width=1.5),
            opacity=0.6
        ))

    # 2. Add Vertical Lines (Vlines)
    # Finding the x-coordinates for the specific ESN
    v_points = points.loc[points['ESN'] == ESN].index
    
    for vp in v_points:
        fig.add_vline(
            x=vp, 
            line_dash="dash", 
            line_color="red", 
            annotation_text=plabel if vp == v_points[0] else "", # Label only the first one
            opacity=0.7
        )

    # 3. Layout and Formatting
    fig.update_layout(
        title=f'Valore {SENSOR} per fase (ESN {ESN})',
        xaxis_title='Cycles',
        yaxis_title='Sensor Value',
        template='plotly_white',
        legend_title="Snapshots",
        width=1000,
        height=600
    )

    # Show and Save
    fig.show()
    fig.write_html(f"{path}/{SENSOR}-{plabel}-plotly.html")


In [ ]:

reload(globals)
import plotly.graph_objects as go
import os

ESN = 101
method = "BY_SNAPSHOT_ALL_SENSORS"
phase = 1

path = f"{globals.PLOT_PATH}/{method}/{ESN}/"
os.makedirs(path, exist_ok=True)

# Define a discrete colorscale (replaces cm.viridis)
colors = [
    '#440154', '#482878', '#3e4989', '#31688e', 
    '#26828e', '#1f9e89', '#6ece58', '#fde725'
]

for sensor in globals.SENSORS.iter():
    filtered = globals.filter(train, "ESN", ESN, [sensor, 'Snapshot'])
    
    # Initialize the figure
    fig = go.Figure()

    # 1. Add the lines for each Snapshot
    for i in range(1, 9):
        df = globals.filter(filtered, "Snapshot", i, [sensor, 'Snapshot'])
        
        fig.add_trace(go.Scatter(
            x=df.index, 
            y=df[sensor],
            mode='lines',
            name=f'Fase {i}',
            line=dict(color=colors[i-1], width=1.5),
            opacity=0.6
        ))

    # 2. Add Vertical Lines (Vlines)
    # Finding the x-coordinates for the specific ESN
    label = ["HPT", "HPC", "WW"]
    vcolors = ["red", "green", "blue"]
    for i, points in enumerate([hpt, hpc, wws]):
        v_points = points.loc[points['ESN'] == ESN].index
        plabel = label[i]
        for vp in v_points:
            fig.add_vline(
                x=vp, 
                line_dash="dash", 
                line_color=vcolors[i], 
                annotation_text=plabel if vp == v_points[0] else "", # Label only the first one
                opacity=0.7
            )

    # 3. Layout and Formatting
    fig.update_layout(
        title=f'Valore {sensor} per fase (ESN {ESN})',
        xaxis_title='Cycles',
        yaxis_title='Sensor Value',
        template='plotly_white',
        legend_title="Snapshots",
        width=1000,
        height=600
    )

    # Show and Save
    fig.show()
    fig.write_html(f"{path}/{sensor}-{plabel}-plotly.html")


In [ ]:
reload(globals)

method = "FFT"

for sensor in globals.SENSORS.iter():
    for esn in globals.ESN:
        path = f"{globals.PLOT_PATH}/{method}/{esn}/"
        os.makedirs(os.path.dirname(path), exist_ok=True)
        plt.figure(figsize=(10, 5))
# Plotting
        l = len(train[sensor])
        np.linspace(0.0, )
        plt.title(f'{sensor} - FFT)')
        plt.xlabel('Frequency')
        plt.ylabel('Amplitude')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.6)
        
        plt.savefig(f"{path}/{SENSOR}-{plabel}-{wsize}")





In [ ]:
for sensor in globals.SENSORS.iter():
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharey=True)
    axes = axes.flatten()
    for i, esn_id in enumerate(train['ESN'].unique()):
        ax = axes[i]

        filtered = train.loc[train['ESN'] == int(esn_id), [sensor]].copy()
        x = filtered[sensor].values

        yf = fft(x)
        t = np.linspace(0, len(x), len(x), endpoint=False)

        ax.plot(t, np.abs(yf), color='darkorange')

        ax.set_title(f'FFT Analysis - ESN {esn_id} {sensor}')
        ax.grid(True, alpha=0.3)
        if i >= 2: ax.set_xlabel('Cicli')
        if i % 2 == 0: ax.set_ylabel('Ampiezza FFT')

    fig.suptitle(f'Confronto Spettrale (FFT): {sensor}', fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95], h_pad=3.0)
    plt.show()